# HHEM-2.1-open — Colab T4 compatibility smoke test (M7)

**Status: PENDING.** This notebook has not been run on a T4. Run it once and attach the report to the PR.

**What it does**
- Uses the merged scorer and its pinned, hash-verified HHEM and FLAN files.
- Checks that HHEM loads with **Transformers 4.56.2**.
- Checks the **7 public examples** on the warm-up and every timed pass against the published scores (tolerance 0.005, same > 0.5 decisions).
- Records the exact Git commit, GPU, driver, CUDA, library versions, model load time, and each example's scores and latencies.

It reads only the public examples. No dev scoring, fine-tuning, or threshold selection.

**Before running:** Use a fresh Colab runtime and choose *Runtime → Change runtime type → T4 GPU*, then connect.

Choose *Runtime → Run all*. If the smoke command reports a failure and Colab stops, run the final two cells manually to inspect and download the verified failure report. A setup failure or a run with no new JSON leaves nothing to download.

In [ ]:
# 1. Confirm the GPU (it must say "Tesla T4").
import subprocess

verified_report = None
expected_commit = None
dependencies_ready = False
subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
    check=True,
)

## 2. Get the code at the requested revision
The default fetches PR #15's current head and verifies the exact commit before running. Set `REF` to a full commit SHA to repeat a specific run, or to `main` after the PR is merged.

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

REF = "refs/pull/15/head"
REPO = "https://github.com/youssefhatembahig6-alt/Enterprise-AI-OS.git"

verified_report = None
expected_commit = None
dependencies_ready = False
repo_dir = Path(tempfile.mkdtemp(prefix="hhem-checkout-", dir="/content"))

def git_commit(revision):
    return subprocess.check_output(
        ["git", "rev-parse", "--verify", revision], cwd=repo_dir, text=True
    ).strip()

subprocess.run(["git", "init", "--quiet"], cwd=repo_dir, check=True)
subprocess.run(["git", "remote", "add", "origin", REPO], cwd=repo_dir, check=True)
subprocess.run(["git", "fetch", "--quiet", "--depth", "1", "origin", REF], cwd=repo_dir, check=True)
requested_commit = git_commit("FETCH_HEAD^{commit}")
subprocess.run(
    ["git", "checkout", "--quiet", "--detach", requested_commit], cwd=repo_dir, check=True
)
if git_commit("HEAD") != requested_commit:
    raise RuntimeError("Checkout did not match the requested revision.")

scorer_dir = repo_dir / "ml" / "hhem-baseline"
if not (scorer_dir / "smoke_model_card.py").is_file():
    raise RuntimeError(f"The smoke script is missing from {requested_commit}.")
expected_commit = requested_commit
print(f"Verified checkout: {expected_commit}")

## 3. Install the pinned libraries
The pinned libraries are `transformers==4.56.2`, `tokenizers==0.22.2` and `safetensors==0.8.0`, among others. torch is Colab's own CUDA build. pip may warn about other Colab packages; that is expected and does not affect this test.

In [ ]:
verified_report = None
dependencies_ready = False
if expected_commit is None:
    raise RuntimeError("Complete the verified checkout before installing dependencies.")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=scorer_dir,
    check=True,
)
dependencies_ready = True

## 4. Run the smoke test
The scorer runs in a fresh Python process and creates a report in a new directory for this attempt. Exit codes: **0** passed, **1** scores or decisions differ, **2** unsupported runtime or versions, **3** download, staging, load, or scoring error. If the command fails and Colab pauses, run the remaining cells manually to download the current failure report.

In [ ]:
verified_report = None
if not dependencies_ready or expected_commit is None:
    raise RuntimeError("Complete the verified checkout and installation before running the smoke test.")
if git_commit("HEAD") != expected_commit:
    raise RuntimeError("The checkout changed after setup.")

report_dir = Path(tempfile.mkdtemp(prefix="hhem-smoke-", dir="/content"))
report_path = report_dir / "hhem-smoke-report.json"
command = [sys.executable, "smoke_model_card.py", "--out", str(report_path)]
completed = subprocess.run(command, cwd=scorer_dir, check=False)
print(f"Smoke script exit code: {completed.returncode}")

if not report_path.is_file():
    completed.check_returncode()
    raise RuntimeError("The smoke process did not produce a report for this attempt.")
report = json.loads(report_path.read_text(encoding="utf-8"))
expected_status = {0: "passed", 1: "failed", 2: "unsupported", 3: "error"}
if (
    report.get("kind") != "m7-hhem-model-card-smoke"
    or report.get("code_commit") != expected_commit
    or completed.returncode not in expected_status
    or report.get("status") != expected_status[completed.returncode]
):
    raise RuntimeError("The report does not match the current commit and smoke exit status.")

verified_report = report_path
print(f"Verified current report: {verified_report}")
completed.check_returncode()

## 5. Inspect the report

In [ ]:
if verified_report is None:
    raise RuntimeError("No verified report was produced by the current attempt.")
r = json.loads(verified_report.read_text(encoding="utf-8"))
print("status:", r["status"], "|", r.get("stage", ""), r.get("reason", ""))
print("code commit:", r["code_commit"])
if "backend" in r:
    b = r["backend"]
    print("GPU:", r["runtime"]["device_name"], "| driver:", r.get("nvidia_driver"))
    print("torch:", b["torch"], "| CUDA:", b["cuda"], "| versions:", r["versions"])
    print("model load (s):", round(r["load_seconds"], 1))
if "latency_per_example" in r:
    print("latency per example:", r["latency_per_example"])
if "comparison" in r:
    c = r["comparison"]
    print("maximum score deviation:", c["max_abs_diff"], "| tolerance:", c["tolerance"])
    for e in c["examples"]:
        samples = e["timed_passes"]
        passes = sum(s["within_tolerance"] and s["decision_matches"] for s in samples)
        print(f"{passes}/{len(samples)} timed passes matched | {e['hypothesis']}")

## 6. Download the report
Attach the verified report from this attempt to the PR, or commit it under `docs/evidence/m7/`. A failed download, staging, load, or scoring attempt may have a failure report even without scores.

In [ ]:
from google.colab import files

if verified_report is None:
    raise RuntimeError("No verified report was produced by the current attempt.")
files.download(str(verified_report))